# Turb-DETR Week 2: Training (Multi-Session Safe v3)

## Fixed issues:
- Labels are REAL FILES, never symlinks
- Uses tar pipe for Drive copies (2 min vs 80 min)
- Archives on Drive reused across sessions
- Checkpoints saved every 5 epochs to Drive
- Auto-resume from any Colab account

## Run order: Every session, run ALL cells top to bottom.


In [1]:
# ============================================================
# CELL 1: Setup
# ============================================================
import os, time, shutil, subprocess
t0_global = time.time()

import torch
if not torch.cuda.is_available():
    raise RuntimeError("NO GPU")
print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install -q ultralytics==8.3.40

from google.colab import drive
drive.mount('/content/drive')

# All persistent paths on Drive
DATASET = "/content/drive/MyDrive/underwater_datasets/trash_ICRA19/dataset"
DRIVE_BASE = "/content/drive/MyDrive/turb_detr_results"
DRIVE_AUGDATA = f"{DRIVE_BASE}/augmented_archives"
DRIVE_CKPT = f"{DRIVE_BASE}/week2_checkpoints"
DRIVE_PREP = f"{DRIVE_BASE}/week2_prep"
DRIVE_RESULTS = f"{DRIVE_BASE}/week2"

if not os.path.exists(DATASET):
    base = "/content/drive/MyDrive/underwater_datasets"
    for n in os.listdir(base):
        if "icra" in n.lower():
            for sub in ["dataset", ""]:
                cand = os.path.join(base, n, sub) if sub else os.path.join(base, n)
                if os.path.exists(os.path.join(cand, "train")):
                    DATASET = cand; break

assert os.path.exists(os.path.join(DATASET, "train")), f"Not found: {DATASET}/train"

for d in [DRIVE_AUGDATA, DRIVE_CKPT, DRIVE_PREP, DRIVE_RESULTS]:
    os.makedirs(d, exist_ok=True)

os.makedirs("/content/turb-detr", exist_ok=True)
os.chdir("/content/turb-detr")
print(f"Dataset: {DATASET}")
print(f"Setup: {time.time()-t0_global:.0f}s")


Mounted at /content/drive
Dataset: /content/drive/MyDrive/underwater_datasets/trash_ICRA19/dataset
Setup: 57s


In [2]:
# ============================================================
# CELL 2: Restore SimAM patch
# ============================================================
import shutil
from pathlib import Path
import ultralytics

PREP = Path(DRIVE_PREP)
ul_mod = Path(ultralytics.__file__).parent / "nn" / "modules"

patched = False
for fname in ["head_patched.py", "head.py", "block_patched.py", "block.py"]:
    src = PREP / fname
    if src.exists():
        dst_name = "head.py" if "head" in fname else "block.py"
        shutil.copy2(src, ul_mod / dst_name)
        print(f"Restored: {fname} -> {dst_name}")
        patched = True; break

if not patched:
    print(f"ERROR: No patch in {PREP}"); print(list(PREP.glob("*")))
    raise FileNotFoundError("Run Week 2A first")

# Verify
for f in ["head.py", "block.py"]:
    txt = (ul_mod / f).read_text()
    if "SimAM" in txt or "simam" in txt:
        print(f"SimAM confirmed in {f}"); break


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Restored: head.py -> head.py
SimAM confirmed in head.py


In [3]:
# ============================================================
# CELL 3: Build dataset (tar pipe for labels, archives for augmented)
# Typical time: 3-5 min (vs 80+ min before)
# ============================================================
import os, subprocess, shutil, time
from pathlib import Path

t0 = time.time()
LOCAL = Path("/content/turb-detr/data")
SRC = Path(DATASET)
ARCHIVES = Path(DRIVE_AUGDATA)
LLOCAL = Path("/content/local_labels")

# ========================
# STEP A: Copy labels with TAR PIPE (fast!) + filter ROV
# ========================
print("Step A: Labels (tar pipe)...")
for split in ["train", "val", "test"]:
    sd = SRC / split
    ll = LLOCAL / split; ll.mkdir(parents=True, exist_ok=True)

    # TAR PIPE: single stream, not 5723 individual file copies
    subprocess.run(
        f"cd {sd} && tar cf - *.txt 2>/dev/null | (cd {ll} && tar xf -)",
        shell=True
    )

    # Filter ROV (class 2) and write to local disk
    lbl_out = LOCAL / f"processed/{split}/labels"
    lbl_out.mkdir(parents=True, exist_ok=True)
    count = 0
    for txt in ll.glob("*.txt"):
        lines = [l.strip() for l in open(txt)
                 if len(l.strip().split()) == 5 and l.strip().split()[0] != "2"]
        with open(lbl_out / txt.name, "w") as f:
            f.write("\n".join(lines) + "\n" if lines else "")
        count += 1
    print(f"  {split}: {count} labels ({time.time()-t0:.0f}s)")

# ========================
# STEP B: Symlink images from Drive
# ========================
print("\nStep B: Images (symlink)...")
for split in ["train", "val", "test"]:
    img_out = LOCAL / f"processed/{split}/images"
    img_out.mkdir(parents=True, exist_ok=True)
    linked = 0
    for f in (SRC / split).glob("*.jpg"):
        d = img_out / f.name
        if not d.exists():
            os.symlink(f.resolve(), d)
            linked += 1
    print(f"  {split}: {linked} linked")

bad = LOCAL / "processed/train/images/obj1658_frame0002383 (1).jpg"
if bad.exists() or bad.is_symlink(): os.unlink(bad)

# ========================
# STEP C: Augmented data — extract from Drive archives
# ========================
HAS_ARCHIVES = (ARCHIVES / "augmented_train.tar.gz").exists()

if HAS_ARCHIVES:
    print("\nStep C: Extracting augmented archives from Drive...")
    for archive in ["augmented_train.tar.gz", "augmented_test.tar.gz"]:
        src = ARCHIVES / archive
        print(f"  {archive}...")
        subprocess.run(f"cd /content/turb-detr && tar xzf {src}", shell=True, check=True)
    print(f"  Extracted ({time.time()-t0:.0f}s)")
else:
    print("\nStep C: NO ARCHIVES FOUND — generating augmented data...")
    print("  This only happens on the very first run.")

    import cv2, numpy as np
    from tqdm import tqdm

    def jaffe(image, c=0.15, depth=3.0, seed=None):
        rng = np.random.RandomState(seed)
        img = image.astype(np.float32) / 255.0
        att = np.exp(-c * depth)
        bs = np.zeros_like(img)
        bs[:,:,0]=0.10*(1-att); bs[:,:,1]=0.30*(1-att); bs[:,:,2]=0.05*(1-att)
        t = img * att + bs
        n = rng.normal(0, 0.02*c*10, img.shape).astype(np.float32)
        return (np.clip(t+n, 0, 1)*255).astype(np.uint8)

    LEVELS = {"light": 0.05, "medium": 0.15, "heavy": 0.30}

    for ds, ds_path in [("train", "processed/train"), ("test", "processed/test")]:
        imgs = sorted((LOCAL / ds_path / "images").glob("*.jpg"))
        print(f"  Augmenting {len(imgs)} {ds} imgs x3...")
        for ln, cv_val in LEVELS.items():
            oi = LOCAL / f"augmented_{ds}/{ln}/images"; oi.mkdir(parents=True, exist_ok=True)
            ol = LOCAL / f"augmented_{ds}/{ln}/labels"; ol.mkdir(parents=True, exist_ok=True)
            prefix = f"{ln}_" if ds == "train" else ""
            for i, ip in enumerate(tqdm(imgs, desc=f"{ds}-{ln}")):
                img = cv2.imread(str(ip))
                if img is None: continue
                cv2.imwrite(str(oi / f"{prefix}{ip.name}"), jaffe(img, c=cv_val, seed=42+i))
                ls = LOCAL / f"{ds_path}/labels/{ip.stem}.txt"
                if ls.exists():
                    shutil.copy2(ls, ol / f"{prefix}{ip.stem}.txt")

    # Save archives to Drive
    print("\n  Saving archives to Drive (one-time)...")
    subprocess.run(f"cd /content/turb-detr && tar czf {ARCHIVES}/augmented_train.tar.gz data/augmented_train/", shell=True)
    subprocess.run(f"cd /content/turb-detr && tar czf {ARCHIVES}/augmented_test.tar.gz data/augmented_test/", shell=True)
    print(f"  Saved to {ARCHIVES}")

# ========================
# STEP D: Build combined_train — REAL FILES for labels, symlinks for images
# ========================
print("\nStep D: Building combined_train...")
LEVELS = {"light": 0.05, "medium": 0.15, "heavy": 0.30}
CIMG = LOCAL / "combined_train/images"
CLBL = LOCAL / "combined_train/labels"

# Wipe any old broken state
if CIMG.exists(): shutil.rmtree(CIMG)
if CLBL.exists(): shutil.rmtree(CLBL)
CIMG.mkdir(parents=True, exist_ok=True)
CLBL.mkdir(parents=True, exist_ok=True)

# Clean images: symlink to Drive (stable)
# Clean labels: COPY to local (real files)
for f in (LOCAL / "processed/train/images").glob("*.jpg"):
    d = CIMG / f.name
    if not d.exists(): os.symlink(f.resolve(), d)

for f in (LOCAL / "processed/train/labels").glob("*.txt"):
    shutil.copy2(f, CLBL / f.name)

# Augmented images: symlink to local extracted files
# Augmented labels: COPY (real files)
for ln in LEVELS:
    aug_img = LOCAL / f"augmented_train/{ln}/images"
    aug_lbl = LOCAL / f"augmented_train/{ln}/labels"
    if not aug_img.exists():
        print(f"  WARNING: {aug_img} not found!"); continue
    for f in aug_img.glob("*.jpg"):
        d = CIMG / f.name
        if not d.exists(): os.symlink(f.resolve(), d)
    for f in aug_lbl.glob("*.txt"):
        shutil.copy2(f, CLBL / f.name)

# Verify
total_img = len(list(CIMG.glob("*")))
total_lbl = len(list(CLBL.glob("*")))
broken = sum(1 for f in CLBL.iterdir() if f.is_symlink())

print(f"\nCombined: {total_img} images, {total_lbl} labels")
print(f"Label symlinks (must be 0): {broken}")
print(f"Total: {time.time()-t0:.0f}s")

if broken > 0:
    raise RuntimeError(f"BROKEN: {broken} label symlinks found!")
print("ALL LABELS ARE REAL FILES.")


Step A: Labels (tar pipe)...
  train: 5723 labels (2527s)
  test: 1144 labels (3404s)

Step B: Images (symlink)...
  train: 5723 linked
  val: 820 linked
  test: 1144 linked

Step C: NO ARCHIVES FOUND — generating augmented data...
  This only happens on the very first run.
  Augmenting 5723 train imgs x3...


train-heavy: 100%|██████████| 5723/5723 [02:57<00:00, 32.33it/s]


  Augmenting 1144 test imgs x3...


test-heavy: 100%|██████████| 1144/1144 [00:36<00:00, 31.60it/s]



  Saving archives to Drive (one-time)...
  Saved to /content/drive/MyDrive/turb_detr_results/augmented_archives

Step D: Building combined_train...

Combined: 22892 images, 22892 labels
Label symlinks (must be 0): 0
Total: 6798s
ALL LABELS ARE REAL FILES.


In [4]:
# ============================================================
# CELL 4: YAML configs
# ============================================================
import yaml, os
from pathlib import Path
os.makedirs("configs/datasets", exist_ok=True)

with open("configs/datasets/trash_icra19_clean.yaml", "w") as f:
    yaml.dump({"path": str(Path("data/processed").resolve()), "train": "train/images",
               "val": "val/images", "test": "test/images", "names": {0: "plastic", 1: "bio"}}, f)

with open("configs/datasets/trash_icra19_augmented.yaml", "w") as f:
    yaml.dump({"path": str(Path("data").resolve()), "train": "combined_train/images",
               "val": "processed/val/images", "test": "processed/test/images",
               "names": {0: "plastic", 1: "bio"}}, f)

for lv in ["light", "medium", "heavy"]:
    with open(f"configs/datasets/trash_icra19_turbid_{lv}.yaml", "w") as f:
        yaml.dump({"path": str(Path(f"data/augmented_test/{lv}").resolve()),
                   "train": "images", "val": "images", "test": "images",
                   "names": {0: "plastic", 1: "bio"}}, f)
print("Configs ready")


Configs ready


In [5]:
import random, numpy as np, torch, os
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
print(f"Seeds = {SEED}")


Seeds = 42


In [6]:
# ============================================================
# CELL 6: Check for checkpoint
# ============================================================
from pathlib import Path
import shutil

CKPT = Path(DRIVE_CKPT)
resume_path = None
training_done = False

if (CKPT / "TRAINING_COMPLETE").exists():
    training_done = True
    print("TRAINING ALREADY COMPLETE. Skipping to evaluation.")
else:
    # Check for last.pt or epoch checkpoints
    for candidate in [CKPT / "turbdetr_augmented_last.pt"] + sorted(CKPT.glob("epoch*.pt"), reverse=True):
        if candidate.exists():
            local_dir = Path("results/turbdetr_augmented/weights")
            local_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidate, local_dir / "last.pt")
            resume_path = str(local_dir / "last.pt")
            print(f"RESUMING from: {candidate.name}")
            break

    if resume_path is None:
        print("No checkpoint. Training from scratch.")


No checkpoint. Training from scratch.


## Training
- Checkpoints every 5 epochs saved to Drive
- Session dies? Rerun notebook — auto-resumes
- 20 epochs total, ~3 hours on T4


In [7]:
# ============================================================
# CELL 7: Train (auto-resume)
# ============================================================
import time, shutil
from pathlib import Path
from ultralytics import RTDETR

if training_done:
    print("Training complete. Skipping.")
else:
    t0 = time.time()
    CKPT = Path(DRIVE_CKPT)

    if resume_path:
        print(f"Resuming from: {resume_path}")
        model = RTDETR(resume_path)
        model.train(resume=True, project="results", name="turbdetr_augmented",
                     device=0, save_period=5)
    else:
        print("Training from scratch (20 epochs)...")
        model = RTDETR("rtdetr-l.pt")
        model.train(data="configs/datasets/trash_icra19_augmented.yaml",
                     epochs=20, imgsz=640, batch=16, seed=42,
                     project="results", name="turbdetr_augmented",
                     device=0, workers=2, patience=10, save_period=5, verbose=True)

    print(f"\nDone ({(time.time()-t0)/60:.1f} min)")

    # Save weights to Drive IMMEDIATELY
    wdir = Path("results/turbdetr_augmented/weights")
    if wdir.exists():
        for w in wdir.glob("*.pt"):
            dst_name = f"turbdetr_augmented_{w.name}" if w.name in ["best.pt","last.pt"] else w.name
            shutil.copy2(w, CKPT / dst_name)
            print(f"  {w.name} -> Drive")

    curves = Path("results/turbdetr_augmented/results.csv")
    if curves.exists():
        shutil.copy2(curves, CKPT / "training_curves.csv")

    (CKPT / "TRAINING_COMPLETE").write_text("done")
    print("TRAINING_COMPLETE marker saved")


Training from scratch (20 epochs)...


100%|██████████| 63.4M/63.4M [00:00<00:00, 107MB/s] 


New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=rtdetr-l.pt, data=configs/datasets/trash_icra19_augmented.yaml, epochs=20, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=5, cache=False, device=0, workers=2, project=results, name=turbdetr_augmented, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed

100%|██████████| 755k/755k [00:00<00:00, 28.5MB/s]


Overriding model.yaml nc=80 with nc=2
WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1     25248  ultralytics.nn.modules.block.HGStem          [3, 32, 48]                   
  1                  -1  6    155072  ultralytics.nn.modules.block.HGBlock         [48, 48, 128, 3, 6]           
  2                  -1  1      1408  ultralytics.nn.modules.conv.DWConv           [128, 128, 3, 2, 1, False]    
  3                  -1  6    839296  ultralytics.nn.modules.block.HGBlock         [128, 96, 512, 3, 6]          
  4                  -1  1      5632  ultralytics.nn.modules.conv.DWConv           [512, 512, 3, 2, 1, False]    
  5                  -1  6   1695360  ultralytics.nn.modules.block.HGBlock         [512, 192, 1024, 5, 6, True, False]
  6                  -1  6   2055808  ultralytics.nn.modules.block.HGBlock         [1024, 192, 1024, 5, 

...

AMP: checks passed ✅


train: Scanning /content/turb-detr/data/combined_train/labels... 22892 images, 612 backgrounds, 0 corrupt: 100%|██████████| 22892/22892 [00:41<00:00, 554.88it/s]


train: New cache created: /content/turb-detr/data/combined_train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/usr/local/lib/python3.12/dist-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /content/turb-detr/data/processed/val/labels... 295 images, 0 backgrounds, 0 corrupt:  36%|███▌      | 295/820 [02:09<03:50,  2.27it/s]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 8: Evaluate on all conditions
# ============================================================
from pathlib import Path
from ultralytics import RTDETR

BEST = None
for p in [Path("results/turbdetr_augmented/weights/best.pt"),
          Path(f"{DRIVE_CKPT}/turbdetr_augmented_best.pt"),
          Path("results/turbdetr_augmented/weights/last.pt"),
          Path(f"{DRIVE_CKPT}/turbdetr_augmented_last.pt")]:
    if p.exists(): BEST = str(p); break

if BEST is None:
    raise FileNotFoundError("No weights found!")

print(f"Using: {BEST}")
model = RTDETR(BEST)

print("\n=== Clean ===")
cr = model.val(data="configs/datasets/trash_icra19_clean.yaml", split="test", device=0)
td_clean = {"map50": cr.box.map50, "map50_95": cr.box.map,
            "prec": cr.box.mp, "rec": cr.box.mr, "ap": list(cr.box.ap50)}
print(f"  mAP@0.5: {td_clean['map50']:.4f}")

td_turbid = {}
for lv in ["light", "medium", "heavy"]:
    print(f"\n=== {lv} ===")
    try:
        r = model.val(data=f"configs/datasets/trash_icra19_turbid_{lv}.yaml", split="test", device=0)
        td_turbid[lv] = {"map50": r.box.map50, "map50_95": r.box.map,
                          "prec": r.box.mp, "rec": r.box.mr, "ap": list(r.box.ap50)}
        print(f"  mAP@0.5: {td_turbid[lv]['map50']:.4f}")
    except Exception as e:
        print(f"  ERROR: {e}")
        td_turbid[lv] = {"map50":0,"map50_95":0,"prec":0,"rec":0,"ap":[0,0]}

print("\nAll evaluations complete")


In [ ]:
# ============================================================
# CELL 9: COMPARISON TABLE
# ============================================================
import csv, os
os.makedirs("results/tables", exist_ok=True)

BL = {"clean": 0.9853, "light": 0.9735, "medium": 0.7368, "heavy": 0.1494}

print()
print("="*80)
print("  VANILLA RT-DETR vs TURB-DETR")
print("="*80)
print(f"  {'Condition':<18} {'Baseline':>12} {'Turb-DETR':>12} {'Delta':>10} {'%':>10}")
print("-"*80)

rows = []
for label, key, val in [("Clean","clean",td_clean),("Light","light",td_turbid.get("light",{})),
                          ("Medium","medium",td_turbid.get("medium",{})),("Heavy","heavy",td_turbid.get("heavy",{}))]:
    bl = BL[key]; td = val.get("map50",0); d = td - bl
    pct = (d/bl*100) if bl > 0 else 0
    print(f"  {label:<18} {bl:>12.4f} {td:>12.4f} {'+' if d>=0 else ''}{d:>9.4f} {'+' if pct>=0 else ''}{pct:>8.1f}%")
    rows.append({"condition":key,"baseline":bl,"turbdetr":td,"delta":d})

print("="*80)

print(f"\n  Per-class AP@0.5 at Heavy:")
td_h = td_turbid.get("heavy",{}).get("ap",[0,0])
for i,(nm,bl_ap) in enumerate([("plastic",0.2155),("bio",0.0834)]):
    ta = td_h[i] if i<len(td_h) else 0
    print(f"    {nm}: {bl_ap:.4f} -> {ta:.4f} ({ta-bl_ap:+.4f})")

hi = td_turbid.get("heavy",{}).get("map50",0) - BL["heavy"]
print("\n" + "="*80)
if hi>0.15: print(f"  EXCELLENT (+{hi:.4f})")
elif hi>0.05: print(f"  GOOD (+{hi:.4f})")
elif hi>0.02: print(f"  MODERATE (+{hi:.4f})")
else: print(f"  WEAK (+{hi:.4f})")
print("="*80)

with open("results/tables/baseline_vs_turbdetr.csv","w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=["condition","baseline","turbdetr","delta"])
    w.writeheader(); w.writerows(rows)
print(f"\nSaved: results/tables/baseline_vs_turbdetr.csv")
print("\n*** SEND THIS TABLE TO CLAUDE ***")


In [ ]:
# ============================================================
# CELL 10: Save to Drive
# ============================================================
import shutil
from pathlib import Path
SAVE = Path(DRIVE_RESULTS); SAVE.mkdir(parents=True, exist_ok=True)

for w in ["best.pt","last.pt"]:
    for s in [Path(f"results/turbdetr_augmented/weights/{w}"), Path(f"{DRIVE_CKPT}/turbdetr_augmented_{w}")]:
        if s.exists(): shutil.copy2(s, SAVE/f"turbdetr_{w}"); print(f"  {w} saved"); break

for f in Path("results/tables").glob("*.csv"):
    shutil.copy2(f, SAVE/f.name); print(f"  {f.name} saved")

for s in [Path("results/turbdetr_augmented/results.csv"), Path(f"{DRIVE_CKPT}/training_curves.csv")]:
    if s.exists(): shutil.copy2(s, SAVE/"turbdetr_curves.csv"); print("  curves saved"); break

print(f"\nAll saved to {SAVE}")
